In [1]:
import json
import os
import sys
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
import torch
from sentence_transformers import SentenceTransformer

# ---- Project path setup ----
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
    
from sparqlce.embedder import SPARQLEmbedder

# Academic plotting theme
sns.set_theme(style="ticks", context="paper", font_scale=1.4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(f"✓ Project root set to: {_PROJECT_ROOT}")

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings


In [2]:
# Seed sampling so Figure 3 is reproducible across machines/runs
random.seed(42)
np.random.seed(42)

import time
import requests

# ====================================================================
# 1. Path Setup
# ====================================================================
_PROJECT_ROOT = Path.cwd().parents[2]

TRIPLETS_PATH = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ1.json") 
RESULTS_PATH = os.path.join(_PROJECT_ROOT, "datasets", "result_sets_dbpedia.json")

# The specific endpoint you provided
SPARQL_ENDPOINT = "https://dbpedia.data.dice-research.org/sparql"

print(f"Project root set to: {_PROJECT_ROOT}")

# ====================================================================
# 2. Load the Local Data & Extract Queries
# ====================================================================
print(f"Loading triplets from {TRIPLETS_PATH}...")
with open(TRIPLETS_PATH, "r", encoding="utf-8") as f:
    triplets_data = json.load(f)

# We must extract the queries FIRST so we know what to execute
anchors = [item.get("query", item.get("anchor_query", "")).strip() for item in triplets_data]

corpus_set = set()
for item in triplets_data:
    q = item.get("query", item.get("anchor_query", "")).strip()
    if q: corpus_set.add(q)
    for p in item.get("positive_queries", []):
        corpus_set.add(p["query"].strip() if isinstance(p, dict) else p.strip())
    for n in item.get("negative_queries", []):
        corpus_set.add(n["query"].strip() if isinstance(n, dict) else n.strip())

corpus_queries = list(corpus_set)
print(f"Found {len(corpus_queries)} unique queries in the test set corpus.")

# ====================================================================
# 3. Execute Queries & Save to JSON
# ====================================================================
def execute_sparql(query, retries=3):
    """Executes a SPARQL query and returns a list of canonical stringified rows."""
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "SPARQL-Evaluation-Script/1.0"
    }
    for attempt in range(retries):
        try:
            response = requests.get(SPARQL_ENDPOINT, params={"query": query}, headers=headers, timeout=45)
            if response.status_code == 200:
                data = response.json()
                results = []
                # Convert each returned binding row into a stable string format for Jaccard calculation
                for binding in data.get("results", {}).get("bindings", []):
                    # Sort keys to guarantee the same row always hashes to the same string
                    row_str = json.dumps(binding, sort_keys=True)
                    results.append(row_str)
                return results
            else:
                print(f"  [Attempt {attempt+1}] Failed with status {response.status_code}")
                time.sleep(2)
        except Exception as e:
            print(f"  [Attempt {attempt+1}] Exception: {e}")
            time.sleep(2)
    return [] # Return empty if it fails after all retries

# Only execute if the file doesn't already exist to save time on reruns
if not os.path.exists(RESULTS_PATH):
    print(f"\nResult file not found. Executing {len(corpus_queries)} queries against DBpedia...")
    query_results = {}
    
    for i, q in enumerate(corpus_queries):
        if i % 10 == 0:
            print(f"Executing query {i}/{len(corpus_queries)}...")
            
        res = execute_sparql(q)
        query_results[q] = res
        time.sleep(0.2) # Polite delay between queries

    # Save to file
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(query_results, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved {len(query_results)} query results to {RESULTS_PATH}")
else:
    print(f"\n{RESULTS_PATH} already exists. Skipping execution phase.")

# ====================================================================
# 4. Load the Cached Results
# ====================================================================
print(f"Loading cached DBpedia results from {RESULTS_PATH}...")
with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    query_results = json.load(f)

print(f"✓ Successfully loaded {len(query_results)} cached query results!")

# ====================================================================
# 5. Stratified Sampling (Jaccard)
# ====================================================================
def build_viz1_pairs(anchors, corpus_queries, result_sets, k_per_anchor=30):
    pairs = []
    # Pre-convert JSON lists to sets and filter out queries that returned 0 results
    cached_sets = {q: set(res) for q, res in result_sets.items() if res}
    
    valid_anchors = [a for a in anchors if a in cached_sets]
    valid_corpus = [c for c in corpus_queries if c in cached_sets]
    
    print(f"Executing stratified sampling for {len(valid_anchors)} valid anchors...")
    
    for a in valid_anchors:
        Ra = cached_sets[a]
        jaccards = []
        for c in valid_corpus:
            if c == a: continue
            Rc = cached_sets[c]
            
            union_len = len(Ra | Rc)
            if union_len == 0: continue
                
            j = len(Ra & Rc) / union_len
            jaccards.append((c, j))

        # Stratified sample across Jaccard bins
        bins = {i: [] for i in range(5)}
        for c, j in jaccards:
            b = min(int(j * 5), 4)
            bins[b].append((c, j))
            
        sampled = []
        per_bin = max(1, k_per_anchor // 5)
        for b in bins.values():
            sampled.extend(random.sample(b, min(per_bin, len(b))))

        for c, j in sampled:
            pairs.append((a, c, j))
            
    return pairs

sampled_pairs = build_viz1_pairs(anchors, corpus_queries, query_results, k_per_anchor=30)
print(f"✓ Stratified sampling complete! Generated {len(sampled_pairs)} pairs for plotting.")

Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings
Loading triplets from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/datasets/dbpedia_testset_RQ1.json...
Found 1677 unique queries in the test set corpus.

/local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/datasets/result_sets_dbpedia.json already exists. Skipping execution phase.
Loading cached DBpedia results from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/datasets/result_sets_dbpedia.json...
✓ Successfully loaded 1677 cached query results!
Executing stratified sampling for 337 valid anchors...
✓ Stratified sampling complete! Generated 6732 pairs for plotting.


In [ ]:
# ====================================================================
# 3. Initialize Models & Encode Corpus
# ====================================================================
import os
import numpy as np
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SPARQLCE_CKPT = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "final_model_mrr_0.4487.pt")
QWEN3_FT_ADAPTER = os.path.join(_PROJECT_ROOT, "checkpoints", "qwen3_square_dbpedia", "best_adapter")
MAX_LEN = 512  # same truncation as fine-tuning

print(f"Using device: {DEVICE}")

class HFEncoder:
    def __init__(self, model_name, prefix=''):
        self.model = SentenceTransformer(model_name, device=DEVICE)
        self.prefix = prefix
    def encode(self, queries):
        texts = [f"{self.prefix}{q}" for q in queries]
        return self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=True)

class SparqlCEEncoderWrap:
    def __init__(self, ckpt):
        self.embedder = SPARQLEmbedder(ckpt)
    def encode(self, queries):
        # HybridSPARQLEncoder.forward already L2-normalizes its output
        embs = self.embedder.encode(queries)
        if not isinstance(embs, np.ndarray): embs = np.array(embs)
        return embs

def _last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_states.shape[0], device=last_hidden_states.device)
    return last_hidden_states[idx, seq_lens]

class Qwen3FinetunedEncoder:
    """Fine-tuned Qwen3-Embedding (LoRA adapter), same pooling as training."""
    def __init__(self, model_name, adapter_path):
        self.tokenizer = AutoTokenizer.from_pretrained(adapter_path, padding_side="left")
        base = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32)
        self.model = PeftModel.from_pretrained(base, adapter_path).to(DEVICE)
        self.model.eval()
    @torch.no_grad()
    def encode(self, queries, batch_size=16):
        out = []
        for i in range(0, len(queries), batch_size):
            tok = self.tokenizer(list(queries[i:i + batch_size]), padding=True,
                                 truncation=True, max_length=MAX_LEN, return_tensors="pt")
            tok = {k: v.to(DEVICE) for k, v in tok.items()}
            h = self.model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
            emb = _last_token_pool(h.last_hidden_state, tok["attention_mask"])
            out.append(F.normalize(emb.float(), p=2, dim=1).cpu().numpy())
        return np.vstack(out)

# Factories: models are loaded ONE AT A TIME and freed after encoding, so all
# six fit on a single GPU (loading them simultaneously caused OOM before).
model_factories = {
    'SparqlCE':  lambda: SparqlCEEncoderWrap(SPARQLCE_CKPT),
    'MiniLM':    lambda: HFEncoder('sentence-transformers/all-MiniLM-L6-v2'),
    'MPNet':     lambda: HFEncoder('sentence-transformers/all-mpnet-base-v2'),
    'E5-base':   lambda: HFEncoder('intfloat/e5-base-v2', prefix='query: '),
    'Qwen3':     lambda: HFEncoder('Qwen/Qwen3-Embedding-0.6B'),
    'Qwen3-FT':  lambda: Qwen3FinetunedEncoder('Qwen/Qwen3-Embedding-0.6B', QWEN3_FT_ADAPTER),
}

unique_needed_queries = list(set([p[0] for p in sampled_pairs] + [p[1] for p in sampled_pairs]))
model_embeddings = {}

print(f"Encoding {len(unique_needed_queries)} unique queries...")
for name, factory in model_factories.items():
    print(f" -> Encoding with {name}...")
    encoder = factory()
    embs = encoder.encode(unique_needed_queries)
    model_embeddings[name] = {q: embs[i] for i, q in enumerate(unique_needed_queries)}
    del encoder
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print("✓ All models encoded.")


In [ ]:
# ====================================================================
# 4. Build Plotting DataFrame & Generate Hexbin Plot (paper-ready)
# ====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# --- 1. Build the DataFrame ---
plot_data = []
for anchor, cand, j_score in sampled_pairs:
    row = {'anchor': anchor, 'candidate': cand, 'jaccard': j_score}
    for name in model_embeddings.keys():
        v1 = model_embeddings[name][anchor]
        v2 = model_embeddings[name][cand]
        # Since embeddings are L2 normalized, dot product == cosine similarity
        row[name] = np.clip(np.dot(v1, v2), -1.0, 1.0)
    plot_data.append(row)

df = pd.DataFrame(plot_data)

# --- 2. Print-ready hexbin panel ---
# Sized for a full-width figure* (~7in wide): fonts keep their point size in
# print instead of being shrunk 2.5x as with the previous 19.5in canvas.
fig, axes = plt.subplots(1, 6, figsize=(7.2, 1.75), sharey=True)

plot_config = [
    ("SparqlCE", "Blues"),   # ours, highlighted
    ("MiniLM",   "Greys"),
    ("MPNet",    "Greys"),
    ("E5-base",  "Greys"),
    ("Qwen3",    "Greys"),
    ("Qwen3-FT", "Greys"),
]

# Draw all hexbins first, then unify the density scale across panels so the
# single colorbar is valid for EVERY panel (same lightness == same count).
# NOTE: with bins="log", matplotlib >= 3.6 keeps raw counts and a LogNorm,
# while older versions store log10(count) with a linear norm - handle both.
from matplotlib.colors import LogNorm

hb_objects = []
for ax, (name, cmap) in zip(axes, plot_config):
    hb = ax.hexbin(df['jaccard'].values, df[name].values,
                   gridsize=30, cmap=cmap, mincnt=1, bins="log")
    hb_objects.append(hb)

vmax = max(float(hb.get_array().max()) for hb in hb_objects)
uses_lognorm = isinstance(hb_objects[0].norm, LogNorm)
for hb in hb_objects:
    hb.set_clim(1.0 if uses_lognorm else 0.0, vmax)

for ax, (name, cmap) in zip(axes, plot_config):
    j = df['jaccard'].values
    c = df[name].values
    rho, _ = spearmanr(j, c)

    ax.plot([0, 1], [0, 1], color="#d62728", linestyle="--",
            linewidth=1.0, zorder=10)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(name, fontsize=8, fontweight="bold", pad=3)
    ax.set_xlabel("Jaccard", fontsize=7)
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_yticks([0, 0.5, 1.0])
    ax.tick_params(labelsize=6, length=2)
    ax.text(0.05, 0.95, fr"$\rho={rho:.2f}$", transform=ax.transAxes,
            fontsize=7, fontweight="bold", va="top",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1.5))
    ax.grid(True, linestyle=":", alpha=0.3, linewidth=0.4)

axes[0].set_ylabel("Cosine similarity", fontsize=7)

# One colorbar, valid for all panels after the shared clim above. Rendered in
# the neutral (Greys) scale; the SparqlCE panel uses the same count scale in blue.
cb = fig.colorbar(hb_objects[1], ax=axes.tolist(), shrink=0.85, pad=0.012, aspect=18)
cb.set_label("pair count (log scale)" if uses_lognorm
             else r"$\log_{10}$(pair count)", fontsize=6.5)
cb.ax.tick_params(labelsize=6)

output_pdf_path = os.path.join(_PROJECT_ROOT, "results", "cosine_vs_jaccard_dbpedia.pdf")
os.makedirs(os.path.dirname(output_pdf_path), exist_ok=True)
plt.savefig(output_pdf_path, bbox_inches="tight")

print(f"Print-ready hexbin plot saved to: {output_pdf_path}")
plt.show()
